<img src="../data/zls.svg" width="120" style="float:right; margin: 0 20px 10px 20px;">

# 03 — Exports and Analysis

Goal: explore the export endpoint and do some basic analysis on STT results.

### Setup

This is the same session → STT → poll pattern from notebook 02. We need a completed transcription to work with.

In [ ]:
import requests
import time
import pandas as pd
from pathlib import Path

BASE_URL = "https://sproochmaschinn.lu"
AUDIO_PATH = Path("../data/sample_audio.wav")

Create a session, upload the audio, and submit an STT request — all in one cell since we've seen these steps before.

In [ ]:
session = requests.post(f"{BASE_URL}/api/session").json()
session_id = session["session_id"]

with open(AUDIO_PATH, "rb") as f:
    stt_response = requests.post(
        f"{BASE_URL}/api/stt/{session_id}",
        files={"audio": f},
        data={"enable_speaker_identification": "true"}
    ).json()

request_id = stt_response["request_id"]
print("STT request_id:", request_id)

Poll until the transcription is ready, then pull out the result.

In [ ]:
MAX_POLLS = 120

for _ in range(MAX_POLLS):
    result = requests.get(f"{BASE_URL}/api/result/{request_id}").json()
    if result["status"] == "completed":
        break
    time.sleep(1)
else:
    raise TimeoutError("STT request did not complete in time.")

transcription = result["result"]

### The export endpoint

The API also provides a dedicated export endpoint that restructures the transcription data — for example, grouped by sentence with timestamps, speakers, and confidence scores.

In [ ]:
export = requests.get(
    f"{BASE_URL}/api/result/{request_id}/export/timestamps",
    params={
        "level": "sentence",
        "include_speakers": "true",
        "include_confidence": "true"
    }
).json()

export

### Comparing raw data vs. export

Here we look at the raw `segments` and `words` from the transcription result side by side, so you can see how they relate to the structured export above.

In [ ]:
segments_df = pd.DataFrame(transcription["segments"])
words_df = pd.DataFrame(transcription["words"])

print("Segments:")
display(segments_df)

print("\nWords (first 50):")
display(words_df.head(50))

### Basic analysis

Each word has a confidence `score` between 0 and 1. Let's see how confident the model was on average across all words.

In [ ]:
if not words_df.empty and "score" in words_df.columns:
    print("Average confidence:", words_df["score"].mean())

## Try it yourself

**Easy:** Change `level` from `"sentence"` to `"word"` in the export call — how does the output change?

**Medium:** Group `words_df` by speaker and count how many words each speaker said. Who talks the most?

**Stretch:** Compute the total speaking time per speaker using the word-level timestamps in `words_df`.